In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 中文显示 and 路径
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

SAVE_DIR = r'credit'
import os
os.makedirs(SAVE_DIR, exist_ok=True)

In [3]:
# ======================== 1. 数据加载与初步探索 ========================
print("="*60)
print("1. 数据加载与初步探索")
print("="*60)

df = pd.read_csv(r'credit/CC GENERAL.csv')
print(f"数据集形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")
print(f"\n数据基本信息:")
print(df.info())
print(f"\n数据统计描述:")
print(df.describe().round(2))

# 缺失值检查
print(f"\n缺失值统计:")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'缺失数量': missing, '缺失比例(%)': missing_pct})
print(missing_df[missing_df['缺失数量'] > 0])

1. 数据加载与初步探索
数据集形状: (8950, 18)
列名: ['CUST_ID', 'BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE', 'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY', 'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE']

数据基本信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8950 entries, 0 to 8949
Data columns (total 18 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   CUST_ID                           8950 non-null   object 
 1   BALANCE                           8950 non-null   float64
 2   BALANCE_FREQUENCY                 8950 non-null   float64
 3   PURCHASES                         8950 non-null   float64
 4   ONEOFF_PURCHASES                  8950 non-null   float64
 5   INSTALLMENTS_PURCHASES            8950 non-null   floa

In [6]:
# ======================== 2. 数据预处理 ========================
print("\n" + "="*60)
print("2. 数据预处理")
print("="*60)

# 复制数据
df_clean = df.copy()

# 2.1 处理缺失值
print(f"MINIMUM_PAYMENTS 缺失数量: {df_clean['MINIMUM_PAYMENTS'].isnull().sum()}")
df_clean['MINIMUM_PAYMENTS'].fillna(df_clean['MINIMUM_PAYMENTS'].median(), inplace=True)
print(f"CREDIT_LIMIT 缺失数量: {df_clean['CREDIT_LIMIT'].isnull().sum()}")
df_clean['CREDIT_LIMIT'].fillna(df_clean['CREDIT_LIMIT'].median(), inplace=True)
print("缺失值填充完成（使用中位数填充）")

# 2.2 删除CUST_ID列（不参与建模）
cust_ids = df_clean['CUST_ID']
df_clean.drop('CUST_ID', axis=1, inplace=True)

# 2.3 异常值处理 — 使用IQR方法检测
print(f"\n异常值检测 (IQR方法):")
outlier_summary = {}
for col in df_clean.columns:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_count = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    outlier_pct = round(outlier_count / len(df_clean) * 100, 2)
    outlier_summary[col] = {'下界': round(lower,2), '上界': round(upper,2), '异常值数量': outlier_count, '比例%': outlier_pct}

outlier_df = pd.DataFrame(outlier_summary).T
print(outlier_df[outlier_df['异常值数量'] > 0].to_string())

# 对异常值进行截尾处理 (Winsorize)
for col in df_clean.columns:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean[col] = df_clean[col].clip(lower, upper)

print(f"\n异常值截尾处理完成")
print(f"处理后数据形状: {df_clean.shape}")


2. 数据预处理
MINIMUM_PAYMENTS 缺失数量: 313
CREDIT_LIMIT 缺失数量: 1
缺失值填充完成（使用中位数填充）

异常值检测 (IQR方法):
                                 下界        上界   异常值数量    比例%
BALANCE                    -2760.51   4942.93   695.0   7.77
BALANCE_FREQUENCY              0.72      1.17  1493.0  16.68
PURCHASES                  -1566.11   2715.87   808.0   9.03
ONEOFF_PURCHASES            -866.11   1443.51  1013.0  11.32
INSTALLMENTS_PURCHASES      -702.96   1171.59   867.0   9.69
CASH_ADVANCE               -1670.73   2784.55  1030.0  11.51
ONEOFF_PURCHASES_FREQUENCY    -0.45      0.75   782.0   8.74
CASH_ADVANCE_FREQUENCY        -0.33      0.56   525.0   5.87
CASH_ADVANCE_TRX              -6.00     10.00   804.0   8.98
PURCHASES_TRX                -23.00     41.00   766.0   8.56
CREDIT_LIMIT               -5750.00  13850.00   248.0   2.77
PAYMENTS                   -1893.51   4177.92   808.0   9.03
MINIMUM_PAYMENTS            -755.93   1715.50   909.0  10.16
PRC_FULL_PAYMENT              -0.21      0.36  1474.0  

In [7]:
# ======================== 3. 探索性数据分析 (EDA) ========================
print("\n" + "="*60)
print("3. 探索性数据分析 (EDA)")
print("="*60)

# 3.1 各特征分布直方图
fig, axes = plt.subplots(6, 3, figsize=(16, 22))
axes = axes.flatten()
for i, col in enumerate(df_clean.columns):
    axes[i].hist(df_clean[col], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('频数')
# 隐藏多余子图
for j in range(len(df_clean.columns), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('各特征分布直方图', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '01_特征分布直方图.png'), dpi=150)
plt.close()
print("图表01: 特征分布直方图 已保存")

# 3.2 相关性热力图
plt.figure(figsize=(16, 12))
corr_matrix = df_clean.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('特征相关性热力图', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '02_相关性热力图.png'), dpi=150)
plt.close()
print("图表02: 相关性热力图 已保存")

# 3.3 购买行为相关特征箱线图
purchase_cols = ['PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'PURCHASES_FREQUENCY']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(purchase_cols):
    ax = axes[i//2, i%2]
    ax.boxplot(df_clean[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(col, fontsize=12)
    ax.set_ylabel('值')
plt.suptitle('购买行为特征箱线图', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '03_购买行为箱线图.png'), dpi=150)
plt.close()
print("图表03: 购买行为箱线图 已保存")

# 3.4 现金预付相关特征箱线图
cash_cols = ['CASH_ADVANCE', 'CASH_ADVANCE_FREQUENCY', 'CASH_ADVANCE_TRX']
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for i, col in enumerate(cash_cols):
    axes[i].boxplot(df_clean[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='lightcoral'))
    axes[i].set_title(col, fontsize=12)
plt.suptitle('现金预付特征箱线图', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '04_现金预付箱线图.png'), dpi=150)
plt.close()
print("图表04: 现金预付箱线图 已保存")

# 3.5 PRC_FULL_PAYMENT分布（全额还款比例）
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(df_clean['PRC_FULL_PAYMENT'], bins=50, color='green', edgecolor='white', alpha=0.8)
axes[0].set_title('全额还款比例分布')
axes[0].set_xlabel('PRC_FULL_PAYMENT')
axes[0].set_ylabel('频数')
axes[1].pie(
    [len(df_clean[df_clean['PRC_FULL_PAYMENT'] == 0]),
     len(df_clean[(df_clean['PRC_FULL_PAYMENT'] > 0) & (df_clean['PRC_FULL_PAYMENT'] < 1)]),
     len(df_clean[df_clean['PRC_FULL_PAYMENT'] == 1])],
    labels=['完全不还(0)', '部分还款(0~1)', '全额还款(1)'],
    autopct='%1.1f%%', colors=['salmon', 'lightskyblue', 'lightgreen']
)
axes[1].set_title('全额还款分类占比')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '05_全额还款分布.png'), dpi=150)
plt.close()
print("图表05: 全额还款分布 已保存")

# 3.6 TENURE分布（信用卡持有期）
plt.figure(figsize=(10, 5))
tenure_counts = df_clean['TENURE'].value_counts().sort_index()
plt.bar(tenure_counts.index, tenure_counts.values, color='teal', edgecolor='white')
plt.title('信用卡持有期(TENURE)分布', fontsize=14)
plt.xlabel('持有期(月)')
plt.ylabel('用户数量')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '06_持有期分布.png'), dpi=150)
plt.close()
print("图表06: 持有期分布 已保存")

# 3.7 余额 vs 购买量散点图
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].scatter(df_clean['BALANCE'], df_clean['PURCHASES'], alpha=0.3, s=5, c='blue')
axes[0].set_xlabel('BALANCE'); axes[0].set_ylabel('PURCHASES')
axes[0].set_title('余额 vs 购买量')
axes[1].scatter(df_clean['CREDIT_LIMIT'], df_clean['PURCHASES'], alpha=0.3, s=5, c='green')
axes[1].set_xlabel('CREDIT_LIMIT'); axes[1].set_ylabel('PURCHASES')
axes[1].set_title('信用额度 vs 购买量')
axes[2].scatter(df_clean['BALANCE'], df_clean['CASH_ADVANCE'], alpha=0.3, s=5, c='red')
axes[2].set_xlabel('BALANCE'); axes[2].set_ylabel('CASH_ADVANCE')
axes[2].set_title('余额 vs 现金预付')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '07_散点图矩阵.png'), dpi=150)
plt.close()
print("图表07: 散点图矩阵 已保存")



3. 探索性数据分析 (EDA)
图表01: 特征分布直方图 已保存
图表02: 相关性热力图 已保存
图表03: 购买行为箱线图 已保存
图表04: 现金预付箱线图 已保存
图表05: 全额还款分布 已保存
图表06: 持有期分布 已保存
图表07: 散点图矩阵 已保存


In [8]:
# ======================== 4. 特征工程 ========================
print("\n" + "="*60)
print("4. 特征工程")
print("="*60)

# 4.1 构造衍生特征
df_fe = df_clean.copy()

# 购买与现金预付比例
df_fe['PURCHASE_RATIO'] = df_fe['PURCHASES'] / (df_fe['PURCHASES'] + df_fe['CASH_ADVANCE'] + 1)
# 信用额度使用率
df_fe['CREDIT_USAGE'] = df_fe['BALANCE'] / (df_fe['CREDIT_LIMIT'] + 1)
# 还款与负债比
df_fe['PAYMENT_TO_BALANCE'] = df_fe['PAYMENTS'] / (df_fe['BALANCE'] + 1)
# 单次购买金额
df_fe['AVG_PURCHASE_PER_TRX'] = df_fe['PURCHASES'] / (df_fe['PURCHASES_TRX'] + 1)
# 单次现金预付金额
df_fe['AVG_CASH_PER_TRX'] = df_fe['CASH_ADVANCE'] / (df_fe['CASH_ADVANCE_TRX'] + 1)
# 分期购买倾向
df_fe['INSTALLMENT_TENDENCY'] = df_fe['INSTALLMENTS_PURCHASES'] / (df_fe['PURCHASES'] + 1)

print("衍生特征构造完成，新增特征:")
new_features = ['PURCHASE_RATIO', 'CREDIT_USAGE', 'PAYMENT_TO_BALANCE',
                'AVG_PURCHASE_PER_TRX', 'AVG_CASH_PER_TRX', 'INSTALLMENT_TENDENCY']
print(new_features)


4. 特征工程
衍生特征构造完成，新增特征:
['PURCHASE_RATIO', 'CREDIT_USAGE', 'PAYMENT_TO_BALANCE', 'AVG_PURCHASE_PER_TRX', 'AVG_CASH_PER_TRX', 'INSTALLMENT_TENDENCY']


In [9]:
# ======================== 5. 数据标准化 ========================
print("\n" + "="*60)
print("5. 数据标准化")
print("="*60)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_fe)
print(f"标准化后数据形状: {X_scaled.shape}")
print(f"均值: {X_scaled.mean():.6f}, 标准差: {X_scaled.std():.6f}")


5. 数据标准化
标准化后数据形状: (8950, 23)
均值: -0.000000, 标准差: 0.978019


In [10]:
# ======================== 6. PCA降维 ========================
print("\n" + "="*60)
print("6. PCA降维分析")
print("="*60)

pca = PCA(n_components=0.95)  # 保留95%方差
X_pca = pca.fit_transform(X_scaled)
print(f"原始特征数: {X_scaled.shape[1]}")
print(f"保留95%方差所需主成分数: {X_pca.shape[1]}")

# PCA解释方差图
pca_full = PCA()
pca_full.fit(X_scaled)
cumsum_var = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1),
            pca_full.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('主成分'); axes[0].set_ylabel('解释方差比例')
axes[0].set_title('各主成分解释方差比例')
axes[1].plot(range(1, len(cumsum_var)+1), cumsum_var, 'o-', color='darkred')
axes[1].axhline(y=0.95, color='gray', linestyle='--', label='95%阈值')
axes[1].axhline(y=0.90, color='gray', linestyle=':', label='90%阈值')
axes[1].set_xlabel('主成分数量'); axes[1].set_ylabel('累计解释方差比例')
axes[1].set_title('累计解释方差比例')
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '08_PCA分析.png'), dpi=150)
plt.close()
print("图表08: PCA分析 已保存")

# 使用2个主成分用于可视化
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)
print(f"前2个主成分解释方差比例: {pca_2d.explained_variance_ratio_.sum():.3f}")


6. PCA降维分析
原始特征数: 23
保留95%方差所需主成分数: 13
图表08: PCA分析 已保存
前2个主成分解释方差比例: 0.496


In [13]:
# ======================== 7. K-Means聚类 ========================
print("\n" + "="*60)
print("7. K-Means聚类分析")
print("="*60)

# 7.1 肘部法则 + 轮廓系数 确定最佳K值
K_range = range(2, 11)
inertias = []
silhouette_scores = []
calinski_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_pca, labels))
    calinski_scores.append(calinski_harabasz_score(X_pca, labels))
    print(f"K={k}: Inertia={km.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.4f}, CH={calinski_scores[-1]:.2f}")

# 绘制K值评估图
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(list(K_range), inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('K值'); axes[0].set_ylabel('Inertia (簇内平方和)')
axes[0].set_title('肘部法则 — 确定最佳K值')
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.5, label='K=4 (肘点)')
axes[0].legend()

axes[1].plot(list(K_range), silhouette_scores, 's-', color='darkgreen', linewidth=2, markersize=8)
axes[1].set_xlabel('K值'); axes[1].set_ylabel('轮廓系数 (Silhouette Score)')
axes[1].set_title('轮廓系数 vs K值')
best_k_sil = list(K_range)[np.argmax(silhouette_scores)]
axes[1].axvline(x=best_k_sil, color='red', linestyle='--', alpha=0.5,
                label=f'最佳K={best_k_sil}')
axes[1].legend()

axes[2].plot(list(K_range), calinski_scores, '^-', color='darkorange', linewidth=2, markersize=8)
axes[2].set_xlabel('K值'); axes[2].set_ylabel('Calinski-Harabasz指数')
axes[2].set_title('CH指数 vs K值')
best_k_ch = list(K_range)[np.argmax(calinski_scores)]
axes[2].axvline(x=best_k_ch, color='red', linestyle='--', alpha=0.5,
                label=f'最佳K={best_k_ch}')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '09_K值评估图.png'), dpi=150)
plt.close()
print("图表09: K值评估图 已保存")

# 7.2 选择最佳K值，构建最终K-Means模型
BEST_K = 4  # 综合肘部法则和轮廓系数
print(f"\n选定最佳K = {BEST_K}")

kmeans = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_pca)
df_fe['Cluster_KMeans'] = kmeans_labels

# K-Means聚类可视化 (PCA 2D) — 用2D投影后的各簇均值作为"中心"
plt.figure(figsize=(10, 8))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#F7DC6F']
centroids_2d_list = []
for i in range(BEST_K):
    mask = kmeans_labels == i
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], c=colors[i],
                label=f'簇 {i}', alpha=0.6, s=10)
    centroids_2d_list.append(X_pca_2d[mask].mean(axis=0))
centroids_2d = np.array(centroids_2d_list)
plt.scatter(centroids_2d[:, 0], centroids_2d[:, 1], c='black', marker='X',
            s=300, edgecolors='white', linewidth=2, label='聚类中心', zorder=5)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
plt.title(f'K-Means聚类结果 (K={BEST_K})', fontsize=14)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '10_KMeans聚类可视化.png'), dpi=150)
plt.close()
print("图表10: K-Means聚类可视化 已保存")


7. K-Means聚类分析
K=2: Inertia=144866.44, Silhouette=0.2455, CH=2689.42
K=3: Inertia=121344.54, Silhouette=0.2401, CH=2472.35
K=4: Inertia=111768.47, Silhouette=0.2052, CH=2044.74
K=5: Inertia=103221.69, Silhouette=0.1905, CH=1845.52
K=6: Inertia=97267.56, Silhouette=0.1972, CH=1676.11
K=7: Inertia=91498.90, Silhouette=0.1859, CH=1578.63
K=8: Inertia=85513.21, Silhouette=0.1946, CH=1537.08
K=9: Inertia=81369.09, Silhouette=0.1978, CH=1470.20
K=10: Inertia=76100.26, Silhouette=0.1881, CH=1465.95
图表09: K值评估图 已保存

选定最佳K = 4
图表10: K-Means聚类可视化 已保存


In [14]:
# ======================== 8. 层次聚类 ========================
print("\n" + "="*60)
print("8. 层次聚类分析")
print("="*60)

# 8.1 绘制层次聚类树状图
plt.figure(figsize=(16, 8))
# 对PCA降维后的数据采样以加快绘制
sample_size = min(2000, len(X_pca))
indices = np.random.choice(len(X_pca), sample_size, replace=False)
X_pca_sample = X_pca[indices]

linked = linkage(X_pca_sample, method='ward')
dendrogram(linked, truncate_mode='level', p=10, leaf_rotation=90,
           leaf_font_size=8, show_contracted=True)
plt.title('层次聚类树状图 (Ward方法)', fontsize=14)
plt.xlabel('样本索引')
plt.ylabel('距离')
plt.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='建议切割线')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '11_层次聚类树状图.png'), dpi=150)
plt.close()
print("图表11: 层次聚类树状图 已保存")

# 8.2 构建层次聚类模型
agg = AgglomerativeClustering(n_clusters=BEST_K)
agg_labels = agg.fit_predict(X_pca)
df_fe['Cluster_Agg'] = agg_labels

# 层次聚类可视化
plt.figure(figsize=(10, 8))
for i in range(BEST_K):
    mask = agg_labels == i
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], c=colors[i],
                label=f'簇 {i}', alpha=0.6, s=10)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
plt.title(f'层次聚类结果 (K={BEST_K})', fontsize=14)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '12_层次聚类可视化.png'), dpi=150)
plt.close()
print("图表12: 层次聚类可视化 已保存")



8. 层次聚类分析
图表11: 层次聚类树状图 已保存
图表12: 层次聚类可视化 已保存


In [15]:
# ======================== 9. DBSCAN聚类 ========================
print("\n" + "="*60)
print("9. DBSCAN聚类分析")
print("="*60)

# 搜索最佳eps参数
from sklearn.neighbors import NearestNeighbors

# K-距离图帮助选择eps
min_pts = 5
neighbors = NearestNeighbors(n_neighbors=min_pts)
neighbors_fit = neighbors.fit(X_pca)
distances, indices = neighbors_fit.kneighbors(X_pca)
distances = np.sort(distances[:, min_pts-1], axis=0)

plt.figure(figsize=(10, 5))
plt.plot(distances, color='steelblue')
plt.xlabel('数据点 (按距离排序)')
plt.ylabel(f'{min_pts}-距离')
plt.title('K-距离图 (用于选择eps参数)')
plt.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, label='eps=1.0')
plt.axhline(y=1.5, color='green', linestyle='--', alpha=0.7, label='eps=1.5')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '13_DBSCAN_K距离图.png'), dpi=150)
plt.close()
print("图表13: DBSCAN K-距离图 已保存")

# 构建DBSCAN模型
dbscan = DBSCAN(eps=1.0, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_pca)
df_fe['Cluster_DBSCAN'] = dbscan_labels

n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_db = list(dbscan_labels).count(-1)
print(f"DBSCAN: 簇数={n_clusters_db}, 噪声点数={n_noise_db} ({n_noise_db/len(dbscan_labels)*100:.1f}%)")

# DBSCAN可视化
plt.figure(figsize=(10, 8))
unique_labels = set(dbscan_labels)
palette = sns.color_palette('husl', len(unique_labels))
for label, color in zip(unique_labels, palette):
    mask = dbscan_labels == label
    if label == -1:
        plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], c='gray',
                    label='噪声点', alpha=0.3, s=5)
    else:
        plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], c=[color],
                    label=f'簇 {label}', alpha=0.6, s=10)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
plt.title(f'DBSCAN聚类结果 (eps=1.0, min_samples=5)', fontsize=14)
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '14_DBSCAN聚类可视化.png'), dpi=150)
plt.close()
print("图表14: DBSCAN聚类可视化 已保存")



9. DBSCAN聚类分析
图表13: DBSCAN K-距离图 已保存
DBSCAN: 簇数=24, 噪声点数=3880 (43.4%)
图表14: DBSCAN聚类可视化 已保存


In [16]:
# ======================== 10. 模型对比评估 ========================
print("\n" + "="*60)
print("10. 模型对比评估")
print("="*60)

evaluation_results = []

# K-Means评估
km_sil = silhouette_score(X_pca, kmeans_labels)
km_db = davies_bouldin_score(X_pca, kmeans_labels)
km_ch = calinski_harabasz_score(X_pca, kmeans_labels)
evaluation_results.append(['K-Means', BEST_K, km_sil, km_db, km_ch])
print(f"K-Means: Silhouette={km_sil:.4f}, Davies-Bouldin={km_db:.4f}, CH={km_ch:.2f}")

# 层次聚类评估
agg_sil = silhouette_score(X_pca, agg_labels)
agg_db = davies_bouldin_score(X_pca, agg_labels)
agg_ch = calinski_harabasz_score(X_pca, agg_labels)
evaluation_results.append(['层次聚类', BEST_K, agg_sil, agg_db, agg_ch])
print(f"层次聚类: Silhouette={agg_sil:.4f}, Davies-Bouldin={agg_db:.4f}, CH={agg_ch:.2f}")

# DBSCAN评估 (排除噪声点)
if n_clusters_db >= 2:
    non_noise_mask = dbscan_labels != -1
    dbscan_sil = silhouette_score(X_pca[non_noise_mask], dbscan_labels[non_noise_mask])
    dbscan_db = davies_bouldin_score(X_pca[non_noise_mask], dbscan_labels[non_noise_mask])
    dbscan_ch = calinski_harabasz_score(X_pca[non_noise_mask], dbscan_labels[non_noise_mask])
else:
    dbscan_sil = dbscan_db = dbscan_ch = np.nan
evaluation_results.append(['DBSCAN', n_clusters_db, dbscan_sil, dbscan_db, dbscan_ch])
print(f"DBSCAN: Silhouette={dbscan_sil:.4f}, Davies-Bouldin={dbscan_db:.4f}, CH={dbscan_ch:.2f}")



10. 模型对比评估
K-Means: Silhouette=0.2052, Davies-Bouldin=1.6158, CH=2044.74
层次聚类: Silhouette=0.1983, Davies-Bouldin=1.6511, CH=1694.20
DBSCAN: Silhouette=-0.3117, Davies-Bouldin=1.0921, CH=29.63


In [17]:
# ======================== 11. 模型评估可视化 ========================
print("\n" + "="*60)
print("11. 模型评估可视化")
print("="*60)

eval_df = pd.DataFrame(evaluation_results, columns=['模型', '簇数', '轮廓系数', 'Davies-Bouldin指数', 'CH指数'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 轮廓系数对比
methods = eval_df['模型'].tolist()
sil_vals = eval_df['轮廓系数'].tolist()
bars = axes[0].bar(methods, sil_vals, color=['steelblue', 'darkgreen', 'darkorange'])
axes[0].set_ylabel('轮廓系数')
axes[0].set_title('模型轮廓系数对比 (越高越好)')
for bar, val in zip(bars, sil_vals):
    if not np.isnan(val):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.4f}', ha='center', fontsize=11)

# DB指数对比
db_vals = eval_df['Davies-Bouldin指数'].tolist()
bars = axes[1].bar(methods, db_vals, color=['steelblue', 'darkgreen', 'darkorange'])
axes[1].set_ylabel('Davies-Bouldin指数')
axes[1].set_title('Davies-Bouldin指数对比 (越低越好)')
for bar, val in zip(bars, db_vals):
    if not np.isnan(val):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.4f}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '15_模型评估对比.png'), dpi=150)
plt.close()
print("图表15: 模型评估对比 已保存")



11. 模型评估可视化
图表15: 模型评估对比 已保存


In [18]:
# ======================== 12. 聚类结果分析 — 用户画像 ========================
print("\n" + "="*60)
print("12. 聚类结果分析 — 用户画像")
print("="*60)

# 使用K-Means结果（最优）进行深度分析
df_fe_original = df_clean.copy()
df_fe_original['Cluster'] = kmeans_labels

# 12.1 每个簇的基本统计信息
cluster_stats = df_fe_original.groupby('Cluster').agg(['mean', 'std']).round(2)
print("\n各簇特征均值:")
print(df_fe_original.groupby('Cluster').mean().round(2))

# 12.2 各簇大小
cluster_sizes = df_fe_original['Cluster'].value_counts().sort_index()
print(f"\n各簇样本数量:\n{cluster_sizes}")
print(f"各簇占比:\n{(cluster_sizes / len(df_fe_original) * 100).round(1)}%")

# 12.3 簇大小饼图
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].pie(cluster_sizes.values, labels=[f'簇{i}\n({v}人)' for i, v in zip(cluster_sizes.index, cluster_sizes.values)],
            autopct='%1.1f%%', colors=colors, explode=(0.02, 0.02, 0.02, 0.02))
axes[0].set_title('各簇用户占比', fontsize=14)

# 12.4 关键特征雷达图
key_features = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'CREDIT_LIMIT',
                'PAYMENTS', 'PURCHASES_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'PRC_FULL_PAYMENT']
cluster_means = df_fe_original.groupby('Cluster')[key_features].mean()
# 归一化到0-1用于雷达图
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

angles = np.linspace(0, 2 * np.pi, len(key_features), endpoint=False).tolist()
angles += angles[:1]  # 闭合

ax = axes[1]
ax = plt.subplot(1, 2, 2, projection='polar')
for i in range(BEST_K):
    values = cluster_means_norm.iloc[i].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'簇{i}', color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(key_features, fontsize=8)
ax.set_title('各簇关键特征雷达图', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '16_簇分布与雷达图.png'), dpi=150)
plt.close()
print("图表16: 簇分布与雷达图 已保存")

# 12.5 关键特征簇间对比柱状图
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()
for i, feature in enumerate(key_features):
    cluster_feature_means = df_fe_original.groupby('Cluster')[feature].mean()
    axes[i].bar(cluster_feature_means.index, cluster_feature_means.values, color=colors)
    axes[i].set_title(feature, fontsize=11)
    axes[i].set_xlabel('簇')
    axes[i].set_ylabel('均值')
plt.suptitle('各簇关键特征均值对比', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '17_簇特征对比柱状图.png'), dpi=150)
plt.close()
print("图表17: 簇特征对比柱状图 已保存")

# 12.6 簇间特征热力图
plt.figure(figsize=(14, 6))
cluster_profile = df_fe_original.groupby('Cluster')[key_features].mean()
sns.heatmap(cluster_profile.T, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('各簇关键特征均值热力图', fontsize=14)
plt.xlabel('簇')
plt.ylabel('特征')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '18_簇特征热力图.png'), dpi=150)
plt.close()
print("图表18: 簇特征热力图 已保存")

# 12.7 各簇TENURE分布对比
plt.figure(figsize=(14, 6))
for i in range(BEST_K):
    subset = df_fe_original[df_fe_original['Cluster'] == i]['TENURE']
    plt.hist(subset, bins=12, alpha=0.5, label=f'簇{i}', color=colors[i])
plt.title('各簇信用卡持有期(TENURE)分布对比', fontsize=14)
plt.xlabel('持有期(月)')
plt.ylabel('用户数量')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, '19_各簇持有期分布.png'), dpi=150)
plt.close()
print("图表19: 各簇持有期分布 已保存")



12. 聚类结果分析 — 用户画像

各簇特征均值:
         BALANCE  BALANCE_FREQUENCY  PURCHASES  ONEOFF_PURCHASES  \
Cluster                                                            
0        1256.72               0.93      77.68             56.37   
1        1657.72               0.98    2162.92           1078.38   
2         468.44               0.89     507.24            203.66   
3        3746.97               0.98     516.56            272.74   

         INSTALLMENTS_PURCHASES  CASH_ADVANCE  PURCHASES_FREQUENCY  \
Cluster                                                              
0                         21.40       1029.67                 0.09   
1                        722.51        280.79                 0.90   
2                        287.01         31.88                 0.59   
3                        193.98       2345.61                 0.35   

         ONEOFF_PURCHASES_FREQUENCY  PURCHASES_INSTALLMENTS_FREQUENCY  \
Cluster                                                              

In [20]:
# ======================== 13. 保存聚类结果 ========================
print("\n" + "="*60)
print("13. 保存聚类结果")
print("="*60)

# 生成用户分群标签
df_result = df[['CUST_ID']].copy()
df_result['Cluster'] = kmeans_labels

# 簇名称映射
cluster_names = {
    0: '低活跃型客户',
    1: '高频购买型客户',
    2: '现金预付型客户',
    3: '稳健消费型客户'
}

# 根据实际数据特征自动命名各簇
cluster_avg = df_fe_original.groupby('Cluster')[['PURCHASES', 'CASH_ADVANCE', 'BALANCE', 'CREDIT_LIMIT']].mean()
print("各簇核心指标均值:")
print(cluster_avg)

# 保存结果
df_result['Cluster_Name'] = df_result['Cluster'].map(cluster_names)
df_result.to_csv(r'credit\clustering_result.csv', index=False)
print("聚类结果已保存至 clustering_result.csv")

# 保存完整带标签数据
df_fe_original.to_csv(r'credit\data_with_clusters.csv', index=False)
print("带聚类标签的完整数据已保存至 data_with_clusters.csv")


13. 保存聚类结果
各簇核心指标均值:
           PURCHASES  CASH_ADVANCE      BALANCE  CREDIT_LIMIT
Cluster                                                      
0          77.680880   1029.674343  1256.717504   2755.086531
1        2162.917110    280.791411  1657.716051   6432.586640
2         507.242003     31.879699   468.440796   3392.083709
3         516.561668   2345.605776  3746.971718   7232.269989
聚类结果已保存至 clustering_result.csv
带聚类标签的完整数据已保存至 data_with_clusters.csv


In [21]:
# ======================== 14. 业务建议总结 ========================
print("\n" + "="*60)
print("14. 业务建议总结")
print("="*60)

for cluster_id in sorted(cluster_names.keys()):
    size = cluster_sizes[cluster_id]
    pct = size / len(df_fe_original) * 100
    print(f"\n{'='*40}")
    print(f"簇{cluster_id}: {cluster_names[cluster_id]} (用户数: {size}, 占比: {pct:.1f}%)")
    print(f"{'='*40}")
    subset = df_fe_original[df_fe_original['Cluster'] == cluster_id]
    print(f"  平均余额: {subset['BALANCE'].mean():.2f}")
    print(f"  平均购买额: {subset['PURCHASES'].mean():.2f}")
    print(f"  平均现金预付: {subset['CASH_ADVANCE'].mean():.2f}")
    print(f"  平均信用额度: {subset['CREDIT_LIMIT'].mean():.2f}")
    print(f"  平均还款额: {subset['PAYMENTS'].mean():.2f}")
    print(f"  平均持有期: {subset['TENURE'].mean():.1f}月")
    print(f"  平均全额还款比例: {subset['PRC_FULL_PAYMENT'].mean():.2f}")


14. 业务建议总结

簇0: 低活跃型客户 (用户数: 2215, 占比: 24.7%)
  平均余额: 1256.72
  平均购买额: 77.68
  平均现金预付: 1029.67
  平均信用额度: 2755.09
  平均还款额: 942.12
  平均持有期: 12.0月
  平均全额还款比例: 0.03

簇1: 高频购买型客户 (用户数: 1810, 占比: 20.2%)
  平均余额: 1657.72
  平均购买额: 2162.92
  平均现金预付: 280.79
  平均信用额度: 6432.59
  平均还款额: 2420.72
  平均持有期: 12.0月
  平均全额还款比例: 0.13

簇2: 现金预付型客户 (用户数: 3597, 占比: 40.2%)
  平均余额: 468.44
  平均购买额: 507.24
  平均现金预付: 31.88
  平均信用额度: 3392.08
  平均还款额: 710.77
  平均持有期: 12.0月
  平均全额还款比例: 0.12

簇3: 稳健消费型客户 (用户数: 1328, 占比: 14.8%)
  平均余额: 3746.97
  平均购买额: 516.56
  平均现金预付: 2345.61
  平均信用额度: 7232.27
  平均还款额: 2282.73
  平均持有期: 12.0月
  平均全额还款比例: 0.03
